# Phase 4 Deterministic Evaluation Walkthrough

This notebook walks through the deterministic evaluation primitives implemented in Phase 4.

It uses the implementation directly:

- `obs_platform.evaluation.registry.DETERMINISTIC_EVALUATORS` exposes the static evaluator registry.
- `ToolExecutionEvaluator`, `StructuredOutputEvaluator`, `TrajectoryEvaluator`, `PolicyEvaluator`, and `EvidenceEvaluator` evaluate an `EvaluationRunView`.
- `ScenarioContract` fixtures under `obs_platform.evaluation.scenario_contracts` drive scenario-aware trajectory and evidence checks.
- `persist_evaluation_result` stores completed evaluator results in `evaluation_results`.

The evaluator examples run from committed telemetry fixtures and do not require a database. The final persistence section uses PostgreSQL.

Prerequisites for the persistence section from the repository root:

```bash
cp .env.example .env
docker compose up -d --wait postgres
uv run alembic upgrade head
```

## 1. Imports And Notebook Helpers

The helpers below are notebook-only display and adapter code. The evaluators, scenario contracts, ingestion service, and persistence function are imported from the application.

In [ ]:
from collections.abc import Iterable
from dataclasses import asdict, is_dataclass
from datetime import UTC, datetime
from decimal import Decimal
import json
from pathlib import Path
import sys
from typing import Any

from IPython.display import Markdown, display
from sqlalchemy import delete, select
from sqlalchemy.ext.asyncio import async_sessionmaker

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from obs_platform.config import DatabaseOnlySettings
from obs_platform.database import create_engine, wait_for_database
from obs_platform.db.models import (
    AgentRun,
    EvaluationResult as EvaluationResultRecord,
    LLMCall,
    Span,
    ToolCall,
)
from obs_platform.evaluation.contracts import SCENARIO_CONTRACTS
from obs_platform.evaluation.evaluators import (
    EvidenceEvaluator,
    PolicyEvaluator,
    StructuredOutputEvaluator,
    ToolExecutionEvaluator,
    TrajectoryEvaluator,
)
from obs_platform.evaluation.persistence import persist_evaluation_result
from obs_platform.evaluation.registry import DETERMINISTIC_EVALUATORS
from obs_platform.evaluation.types import EvaluationRunView
from obs_platform.ingestion.runs import ingest_run_event
from obs_platform.telemetry.v1 import ExtendedRunEvent, load_fixture


DEMO_RUN_ID = "notebook-phase-4-evaluation-demo"


def jsonable(value: Any) -> Any:
    if is_dataclass(value):
        return asdict(value)
    if hasattr(value, "model_dump"):
        return value.model_dump(mode="json")
    if isinstance(value, Decimal):
        return float(value)
    if isinstance(value, datetime):
        return value.isoformat()
    if isinstance(value, list):
        return [jsonable(item) for item in value]
    if isinstance(value, dict):
        return {key: jsonable(item) for key, item in value.items()}
    return value


def table(rows: Iterable[dict[str, Any]]) -> None:
    rows = list(rows)
    if not rows:
        display(Markdown("_No rows._"))
        return
    headers = list(rows[0])
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    for row in rows:
        values = [json.dumps(jsonable(row.get(header)), sort_keys=True) for header in headers]
        lines.append("| " + " | ".join(values) + " |")
    display(Markdown("\n".join(lines)))


def result_row(evaluator_name: str, result: Any) -> dict[str, Any]:
    return {
        "evaluator": evaluator_name,
        "passed": result.passed,
        "score": result.score,
        "label": result.label,
        "severity": result.severity,
        "reason": result.reason,
        "findings": [finding.code for finding in result.findings],
    }


def finding_rows(result: Any) -> list[dict[str, Any]]:
    return [
        {"code": finding.code, "message": finding.message, "data": finding.data}
        for finding in result.findings
    ]


def view_from_event(event: ExtendedRunEvent) -> EvaluationRunView:
    return EvaluationRunView(
        run_id=event.run_id,
        schema_version=event.schema_version,
        event_type=event.event_type,
        agent_name=event.agent_name,
        agent_version=event.agent_version,
        prompt_version=event.prompt_version,
        environment=event.environment,
        raw_input=event.raw_input,
        normalized_input=event.normalized_input,
        scenario_id=event.scenario_id,
        started_at=event.started_at,
        completed_at=event.completed_at,
        status=event.status,
        execution_latency_ms=event.execution_latency_ms,
        wall_clock_duration_ms=event.wall_clock_duration_ms,
        resume_count=event.resume_count,
        hitl_required=event.hitl.required,
        hitl_state=event.hitl.state,
        hitl_checkpoint_id=event.hitl.checkpoint_id,
        hitl_decision=event.hitl.decision,
        hitl_requested_at=event.hitl.requested_at,
        hitl_decided_at=event.hitl.decided_at,
        hitl_pending_action=event.hitl.pending_action,
        usage_total_llm_calls=event.usage.total_llm_calls,
        usage_total_tool_calls=event.usage.total_tool_calls,
        usage_total_tokens=event.usage.total_tokens,
        usage_total_retries=event.usage.total_retries,
        usage_total_estimated_cost_usd=event.usage.total_estimated_cost_usd,
        final_result_output=(event.final_result.output if event.final_result else None),
        final_result_source_references=(
            event.final_result.source_references if event.final_result else None
        ),
        runtime_error_category=(event.runtime_error.category if event.runtime_error else None),
        runtime_error_code=(event.runtime_error.code if event.runtime_error else None),
        runtime_error_message=(event.runtime_error.message if event.runtime_error else None),
        runtime_error_failed_component=(
            event.runtime_error.failed_component if event.runtime_error else None
        ),
        spans=[
            {
                "span_id": span.span_id,
                "parent_span_id": span.parent_span_id,
                "name": span.name,
                "sequence": span.sequence,
                "started_at": span.started_at,
                "completed_at": span.completed_at,
                "status": span.status,
                "input": span.input,
                "output": span.output,
                "metadata": span.metadata,
                "error_category": span.error.category if span.error else None,
                "error_code": span.error.code if span.error else None,
                "error_message": span.error.message if span.error else None,
                "error_failed_component": span.error.failed_component if span.error else None,
            }
            for span in event.spans
        ],
        tool_calls=[
            {
                "tool_call_id": tool_call.tool_call_id,
                "span_id": tool_call.span_id,
                "tool_name": tool_call.tool_name,
                "sequence": tool_call.sequence,
                "arguments": tool_call.arguments,
                "result": tool_call.result,
                "started_at": tool_call.started_at,
                "completed_at": tool_call.completed_at,
                "latency_ms": tool_call.latency_ms,
                "retry_count": tool_call.retry_count,
                "status": tool_call.status,
                "error_category": tool_call.error.category if tool_call.error else None,
                "error_code": tool_call.error.code if tool_call.error else None,
                "error_message": tool_call.error.message if tool_call.error else None,
                "error_failed_component": (
                    tool_call.error.failed_component if tool_call.error else None
                ),
            }
            for tool_call in event.tool_calls
        ],
        llm_calls=[
            {
                "llm_call_id": llm_call.llm_call_id,
                "span_id": llm_call.span_id,
                "sequence": index,
                "call_type": llm_call.call_type,
                "model": llm_call.model,
                "provider": llm_call.provider,
                "started_at": llm_call.started_at,
                "completed_at": llm_call.completed_at,
                "latency_ms": llm_call.latency_ms,
                "prompt_tokens": llm_call.prompt_tokens,
                "completion_tokens": llm_call.completion_tokens,
                "total_tokens": llm_call.total_tokens,
                "estimated_cost_usd": llm_call.estimated_cost_usd,
                "input_payload": llm_call.input_payload,
                "output_payload": llm_call.output_payload,
                "status": llm_call.status,
                "error_category": llm_call.error.category if llm_call.error else None,
                "error_code": llm_call.error.code if llm_call.error else None,
                "error_message": llm_call.error.message if llm_call.error else None,
                "error_failed_component": (
                    llm_call.error.failed_component if llm_call.error else None
                ),
            }
            for index, llm_call in enumerate(event.llm_calls, start=1)
        ],
    )


def copy_run(run: EvaluationRunView, **updates: Any) -> EvaluationRunView:
    payload = run.model_dump()
    payload.update(updates)
    return EvaluationRunView.model_validate(payload)


def clone_fixture_view(fixture_name: str, run_id: str | None = None) -> EvaluationRunView:
    view = view_from_event(load_fixture(fixture_name))
    return copy_run(view, run_id=run_id) if run_id else view


## 2. Evaluator Registry And Result Shape

Phase 4 uses an explicit static registry. Each evaluator has stable class metadata, and all evaluators return the same `EvaluationResult` shape.

In [ ]:
table([
    {
        "name": evaluator.name,
        "version": evaluator.version,
        "type": evaluator.type.value,
        "class": evaluator.__class__.__name__,
    }
    for evaluator in DETERMINISTIC_EVALUATORS
])

## 3. Baseline: Run Every Evaluator Against A Healthy Fixture

The healthy success fixture has successful tool calls and a non-empty final result. It is not tied to a Phase 4 scenario contract, so trajectory and evidence checks are intentionally `not_applicable`.

In [ ]:
healthy_run = clone_fixture_view("healthy_success")
healthy_results = [
    (evaluator.name, evaluator.evaluate(healthy_run))
    for evaluator in DETERMINISTIC_EVALUATORS
]

table(result_row(name, result) for name, result in healthy_results)

## 4. Tool Execution Evaluator

`ToolExecutionEvaluator` scores successful tool calls over total tool calls. `failure` and `error` both fail the evaluator, but the finding preserves the exact tool status.

In [ ]:
tool_failure_run = clone_fixture_view("tool_failure")
tool_execution_result = ToolExecutionEvaluator().evaluate(tool_failure_run)

table([result_row("tool_execution", tool_execution_result)])
table(finding_rows(tool_execution_result))

The zero-tool-call edge case passes vacuously and has no score because there is no denominator.

In [ ]:
zero_tool_run = copy_run(
    healthy_run,
    run_id="notebook-zero-tool-run",
    tool_calls=[],
    usage_total_tool_calls=0,
)
zero_tool_result = ToolExecutionEvaluator().evaluate(zero_tool_run)

table([result_row("tool_execution", zero_tool_result)])

## 5. Structured Output Evaluator

`StructuredOutputEvaluator` checks only whether a successful run has a non-empty final result output. It does not inspect producer-specific keys or validate source references.

In [ ]:
non_empty_output_result = StructuredOutputEvaluator().evaluate(healthy_run)
empty_output_run = copy_run(
    healthy_run,
    run_id="notebook-empty-output-run",
    final_result_output={},
)
empty_output_result = StructuredOutputEvaluator().evaluate(empty_output_run)
runtime_error_result = StructuredOutputEvaluator().evaluate(
    clone_fixture_view("unsupported_claim_candidate")
)

table([
    result_row("structured_output/non_empty", non_empty_output_result),
    result_row("structured_output/empty", empty_output_result),
    result_row("structured_output/not_applicable", runtime_error_result),
])
table(finding_rows(empty_output_result))

## 6. Scenario Contracts

`TrajectoryEvaluator` and `EvidenceEvaluator` use machine-readable scenario contracts. Phase 4 currently ships one real golden scenario contract and one synthetic debug contract.

In [ ]:
table([
    {
        "scenario_id": scenario_id,
        "required_tools": contract.required_tools,
        "forbidden_tools": contract.forbidden_tools,
        "ordering_constraints": contract.ordering_constraints,
        "terminal": contract.terminal,
        "required_evidence": contract.required_evidence,
        "expected_asset_identity": contract.expected_asset_identity,
    }
    for scenario_id, contract in SCENARIO_CONTRACTS.items()
])

## 7. Trajectory Evaluator

`TrajectoryEvaluator` checks required tools, forbidden tools, pairwise ordering constraints, and optional terminal conditions for recognized scenario IDs. Unknown or missing scenario IDs are `not_applicable`.

In [ ]:
hitl_pending_result = TrajectoryEvaluator().evaluate(clone_fixture_view("hitl_pending"))
hitl_approved_result = TrajectoryEvaluator().evaluate(clone_fixture_view("hitl_approved"))
trajectory_error_result = TrajectoryEvaluator().evaluate(
    clone_fixture_view("trajectory_error")
)
not_applicable_trajectory_result = TrajectoryEvaluator().evaluate(healthy_run)

table([
    result_row("trajectory/hitl_pending", hitl_pending_result),
    result_row("trajectory/hitl_approved", hitl_approved_result),
    result_row("trajectory/debug_violation", trajectory_error_result),
    result_row("trajectory/no_contract", not_applicable_trajectory_result),
])
table(finding_rows(trajectory_error_result))

## 8. Policy Evaluator

`PolicyEvaluator` applies to every run. It deterministically flags unauthorized consequential actions and asset-specific calls after an `unknown_asset` span. It is the only Phase 4 evaluator that sets result-level severity.

In [ ]:
policy_fixture_result = PolicyEvaluator().evaluate(clone_fixture_view("policy_violation"))

timestamp = datetime(2026, 8, 29, 12, 0, tzinfo=UTC)
unknown_asset_run = copy_run(
    healthy_run,
    run_id="notebook-unknown-asset-policy-run",
    scenario_id=None,
    spans=[
            {
                "span_id": "span-unknown-asset",
                "parent_span_id": None,
                "name": "unknown_asset",
                "sequence": 2,
                "started_at": timestamp,
                "completed_at": timestamp,
                "status": "success",
                "input": None,
                "output": None,
                "metadata": None,
                "error_category": None,
                "error_code": None,
                "error_message": None,
                "error_failed_component": None,
            }
    ],
    tool_calls=[
            {
                "tool_call_id": "tool-after-unknown-asset",
                "span_id": "span-unknown-asset",
                "tool_name": "get_asset_status",
                "sequence": 3,
                "arguments": {"asset_id": "unknown"},
                "result": {},
                "started_at": timestamp,
                "completed_at": timestamp,
                "latency_ms": 20,
                "retry_count": 0,
                "status": "success",
                "error_category": None,
                "error_code": None,
                "error_message": None,
                "error_failed_component": None,
            }
    ],
)
unknown_asset_result = PolicyEvaluator().evaluate(unknown_asset_run)

table([
    result_row("policy/unauthorized_submit", policy_fixture_result),
    result_row("policy/unknown_asset", unknown_asset_result),
])
table(finding_rows(policy_fixture_result) + finding_rows(unknown_asset_result))

## 9. Evidence Evaluator

`EvidenceEvaluator` checks that every required evidence ID in a recognized scenario contract appears in `final_result_source_references`. Current Phase 4 contracts have no required evidence IDs, so the first example passes with no findings. The second example mutates the in-memory contract to demonstrate the missing-evidence path.

In [ ]:
gs08_evidence_run = clone_fixture_view("hitl_approved")
gs08_evidence_result = EvidenceEvaluator().evaluate(gs08_evidence_run)
no_contract_evidence_result = EvidenceEvaluator().evaluate(healthy_run)

original_contract = SCENARIO_CONTRACTS["GS-08"]
try:
    SCENARIO_CONTRACTS["GS-08"] = original_contract.model_copy(
        update={"required_evidence": ["policy-pp-002", "maintenance-history-pump-103"]}
    )
    missing_evidence_result = EvidenceEvaluator().evaluate(gs08_evidence_run)
finally:
    SCENARIO_CONTRACTS["GS-08"] = original_contract

table([
    result_row("evidence/gs08_current_contract", gs08_evidence_result),
    result_row("evidence/no_contract", no_contract_evidence_result),
    result_row("evidence/synthetic_missing_required", missing_evidence_result),
])
table(finding_rows(missing_evidence_result))

## 10. Persist Evaluation Results

The persistence function stores one completed `evaluation_results` row per evaluator invocation. This section ingests a demo run, evaluates it with the registry, persists the results, and inspects the stored rows.

This section requires the repository's PostgreSQL service and migrations.

In [ ]:
settings = DatabaseOnlySettings()
engine = create_engine(settings.db)
Session = async_sessionmaker(engine, expire_on_commit=False)
await wait_for_database(engine)

async def delete_demo_run() -> None:
    async with Session() as session:
        await session.execute(
            delete(EvaluationResultRecord).where(
                EvaluationResultRecord.run_id == DEMO_RUN_ID
            )
        )
        await session.execute(delete(ToolCall).where(ToolCall.run_id == DEMO_RUN_ID))
        await session.execute(delete(LLMCall).where(LLMCall.run_id == DEMO_RUN_ID))
        await session.execute(delete(Span).where(Span.run_id == DEMO_RUN_ID))
        await session.execute(delete(AgentRun).where(AgentRun.run_id == DEMO_RUN_ID))
        await session.commit()

await delete_demo_run()
display(Markdown("Database connection ready and previous demo rows removed."))

In [ ]:
demo_event = load_fixture("policy_violation").model_copy(update={"run_id": DEMO_RUN_ID})
demo_run_view = view_from_event(demo_event)

async with Session() as session:
    await ingest_run_event(session, demo_event)
    persisted_records = []
    for evaluator in DETERMINISTIC_EVALUATORS:
        result = evaluator.evaluate(demo_run_view)
        record = await persist_evaluation_result(
            session,
            demo_run_view.run_id,
            evaluator,
            result,
        )
        persisted_records.append(record)

table([
    {
        "id": record.id,
        "run_id": record.run_id,
        "evaluator_name": record.evaluator_name,
        "status": record.status,
        "score": record.score,
        "label": record.label,
        "severity": record.severity,
        "reason": record.reason,
        "findings": record.findings,
        "regression_run_id": record.regression_run_id,
    }
    for record in persisted_records
])

Persisting the same evaluator result again inserts another row. Phase 4 deliberately does not upsert evaluator results.

In [ ]:
async with Session() as session:
    evaluator = ToolExecutionEvaluator()
    result = evaluator.evaluate(demo_run_view)
    duplicate_record = await persist_evaluation_result(
        session,
        demo_run_view.run_id,
        evaluator,
        result,
    )
    rows = (
        await session.execute(
            select(
                EvaluationResultRecord.id,
                EvaluationResultRecord.evaluator_name,
                EvaluationResultRecord.evaluator_version,
                EvaluationResultRecord.status,
                EvaluationResultRecord.created_at,
            )
            .where(EvaluationResultRecord.run_id == DEMO_RUN_ID)
            .order_by(EvaluationResultRecord.id)
        )
    ).all()

table([dict(row._mapping) for row in rows])

## 11. Cleanup

This removes only the notebook demo run and its persisted evaluation rows.

In [ ]:
await delete_demo_run()
await engine.dispose()
display(Markdown("Notebook demo rows removed and engine disposed."))